# Bayesian Optimisation — Function 2 (2D)

**Author:** Dibyajyoti Pradhan  
**Programme:** Imperial College London Professional Certificate in ML/AI  
**Module:** BBO Capstone (Modules 12–22)  

**Strategy note:** Moderate, gradually improving. Continue directional drift.

## 1. Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# Add src/ to path for reusable utilities
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))
from bbo_utils import (
    lhs_candidates, fit_gp, gp_posterior,
    ucb_acquisition, suggest_next_query,
    dimension_sensitivity, plot_running_best,
    plot_gp_surface_2d,
)

np.random.seed(42)

## 2. Load Initial Data

Initial observations provided at the start of the BBO challenge.

In [ ]:
data_dir = os.path.join('..', 'data', 'initial_data', 'function_2')

X_init = np.load(os.path.join(data_dir, 'initial_inputs.npy'))
y_init = np.load(os.path.join(data_dir, 'initial_outputs.npy'))

print(f'Initial inputs shape:  {X_init.shape}')
print(f'Initial outputs shape: {y_init.shape}')
print(f'Output range: [{y_init.min():.6f}, {y_init.max():.6f}]')
print(f'Best initial output:   {y_init.max():.6f}')
print(f'Best initial input:    {X_init[np.argmax(y_init)]}')

## 3. Add Subsequent Query Data

Queries from Weeks 1–5 with confirmed outputs.  
Week 6 submitted — awaiting results (not included in training).

In [ ]:
# Weekly queries — Weeks 1–5 (confirmed outputs)
queries_X = np.array([
    [0.001000, 0.999000],   # W1
    [0.788596, 0.912389],   # W2
    [0.642398, 0.967781],   # W3
    [0.670079, 0.926276],   # W4
    [0.670333, 0.923855],   # W5
])
queries_y = np.array([
    -0.103332016167505,   # W1
     0.003541507816049,   # W2
     0.283503040504399,   # W3
     0.435473441247849,   # W4
     0.500916018237762,   # W5
])

# Week 6 submitted — awaiting result (excluded from training)
# x_w6 = np.array([0.668952, 0.918290])

X_train = np.vstack([X_init, queries_X])
y_train = np.append(y_init, queries_y)

print(f'Total observations: {len(y_train)}')
print(f'Best observed output: {y_train.max():.6f}')
print(f'Best observed input:  {X_train[np.argmax(y_train)]}')

## 4. Data Exploration

In [ ]:
print('Output statistics:')
print(f'  Min:  {y_train.min():.6f}')
print(f'  Max:  {y_train.max():.6f}')
print(f'  Mean: {y_train.mean():.6f}')
print(f'  Std:  {y_train.std():.6f}')

# Running best across all observations
plot_running_best(y_train, title='Function 2 — Running Best Output')

In [ ]:
# 2D scatter of observed points
plt.figure(figsize=(7, 6))
sc = plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train,
                 cmap='viridis', s=80, edgecolors='k')
plt.colorbar(sc, label='Output value')
plt.xlabel('X₁'); plt.ylabel('X₂')
plt.title('Function 2 — Observed Query Points')
plt.grid(True); plt.tight_layout(); plt.show()

## 5. Bayesian Optimisation (UCB, β=0.5)

Strategy: exploitation-heavy (trust region ±0.08)

In [ ]:
next_query, fitted_model = suggest_next_query(
    X_train=X_train,
    y_train=y_train,
    n_candidates=500_000,
    beta=0.5,
    length_scale=0.1,
    noise_level=1e-6,
    fit_noise=True,
    trust_region_radius=0.08,
    seed=42,
)

# Format to 6 decimal places (BBO portal requirement)
coords = ', '.join(f'{v:.6f}' for v in next_query)
print(f'Week 7 suggested query for Function 2:')
print(f'  ({coords})')

In [ ]:
# Visualise GP posterior
plot_gp_surface_2d(
    model=fitted_model,
    X_train=X_train,
    y_train=y_train,
    next_query=next_query,
    title='Function 2',
)

## 6. Week 7 Strategy Summary

| Field | Value |
|-------|-------|
| Function | F2 (2D) |
| Total observations | n/a (computed above) |
| Acquisition | UCB, β=0.5 |
| Trust region | ±0.08 per dim |
| Strategy | Moderate, gradually improving. Continue directional drift. |

> Submit the coordinates printed above via the BBO portal. All inputs must be in `[0, 1)` to 6 decimal places.